# CSET419 – Introduction to Generative AI
## Lab 11 – Fine-Tuning GPT-2 for Real-World Applications

**Objective:** Fine-tune GPT-2 for two real-world use cases:
- **Component I** – Product Review Generator (E-Commerce)
- **Component II** – Recipe Instruction Generator (Food-Tech)

> ⚠️ **Before running:** Go to `Runtime → Change runtime type → T4 GPU` for faster training.

## Step 0 – Install Required Libraries

In [1]:
!pip install transformers datasets accelerate -q

## Step 1 – Import Libraries & Setup

In [2]:
import torch
import math
import warnings
import os

os.environ["TOKENIZERS_PARALLELISM"] = "false"
warnings.filterwarnings("ignore")

from transformers import (
    GPT2LMHeadModel,
    GPT2Tokenizer,
    Trainer,
    TrainingArguments,
    DataCollatorForLanguageModeling,
    set_seed
)
from datasets import Dataset

set_seed(42)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")
print("All libraries loaded successfully!")

Using device: cuda
All libraries loaded successfully!


## Helper Function – Text Generation

In [3]:
def generate_text(model, tokenizer, prompt, max_length=100):
    """Generate text from a given prompt using the model."""
    model.eval()
    inputs = tokenizer.encode(prompt, return_tensors='pt').to(model.device)
    with torch.no_grad():
        out = model.generate(
            inputs,
            max_length=max_length,
            temperature=0.8,
            top_k=50,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id
        )
    return tokenizer.decode(out[0], skip_special_tokens=True)

---
# 🛒 COMPONENT I – Product Review Generator (E-Commerce)

**Scenario:** You are an AI engineer at an e-commerce company. Fine-tune GPT-2 to auto-generate realistic product reviews.

### Step 1 – Load GPT-2 Model

In [4]:
print("Loading GPT-2 model and tokenizer...")

tokenizer = GPT2Tokenizer.from_pretrained('gpt2')
model = GPT2LMHeadModel.from_pretrained('gpt2')

tokenizer.pad_token = tokenizer.eos_token
model.config.pad_token_id = tokenizer.eos_token_id

print(f"Model loaded! Parameters: {sum(p.numel() for p in model.parameters()):,}")

Loading GPT-2 model and tokenizer...


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Model loaded! Parameters: 124,439,808


### Step 2 – Generate Baseline Reviews (Before Fine-Tuning)

In [5]:
review_prompts = [
    'This product is',
    'I bought this phone and',
    'The quality of this item',
]

print("=" * 60)
print("  BASELINE REVIEWS (Before Fine-Tuning)")
print("=" * 60)

baseline = {}
for p in review_prompts:
    baseline[p] = generate_text(model, tokenizer, p)
    print(f"\nPrompt : {p}")
    print(f"Output : {baseline[p]}")

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


  BASELINE REVIEWS (Before Fine-Tuning)

Prompt : This product is
Output : This product is made from high quality, lightweight stainless steel. If you are looking for something a little more durable, it's a good choice.

Laser Pouch

Not all of our laser printers are created equal. We have a laser printer that comes with all of our printer parts. These parts include our new 3D printer and a 3D printed printing service. All of our printers make laser printers, including our laser printers, using laser technology. Our laser printers are the most

Prompt : I bought this phone and
Output : I bought this phone and I have not used it on a lot of people. I have also not used it on any other people.

The screen was amazing and the sound was amazing. It was not loud. I would never use it on a tv, laptop, smartphone or other connected device in the future.

The battery life is good. The phone works great but it has so many problems.

I have been using phones that have the Snapdragon 616 processo

### Step 3 – Prepare Dataset & Fine-Tune

In [6]:
corpus = [
    'this phone has an amazing battery life and the camera quality is outstanding for the price.',
    'i bought this laptop for college and it handles all my assignments and coding projects perfectly.',
    'the sound quality of these headphones is incredible with deep bass and clear vocals.',
    'this smartwatch tracks my steps accurately and the heart rate monitor is very reliable.',
    'great wireless earbuds with noise cancellation that blocks out all background sound.',
    'the keyboard feels very comfortable for long typing sessions and the backlight is a nice touch.',
    'this portable charger saved me during travel and it charges my phone three times on a single charge.',
    'the tablet screen is bright and colorful which makes watching movies a great experience.',
    'i love this fitness tracker because it motivates me to reach my daily exercise goals.',
    'this bluetooth speaker is compact but delivers surprisingly loud and clear audio.',
    'the delivery was fast and the product was packed securely with no damage at all.',
    'excellent value for money and the build quality feels premium despite the affordable price.',
    'the customer service team was very helpful when i had questions about the product features.',
    'this camera takes stunning photos in low light and the video recording quality is very smooth.',
    'i have been using this product for three months and it still works perfectly like day one.',
    'the design is sleek and modern and it looks great on my desk next to my other gadgets.',
    'easy to set up right out of the box and the instructions were clear and simple to follow.',
    'highly recommend this product to anyone looking for quality and reliability at a fair price.',
    'the software updates keep adding new features which makes this purchase even more worthwhile.',
    'best purchase i made this year and i would definitely buy from this brand again.',
]

dataset = Dataset.from_dict({'text': corpus})
tokenized = dataset.map(
    lambda x: tokenizer(x['text'], truncation=True, max_length=128, padding='max_length'),
    batched=True,
    remove_columns=['text']
)
split = tokenized.train_test_split(test_size=0.15, seed=42)
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

print(f"Dataset prepared: {len(split['train'])} train, {len(split['test'])} test samples")

Map:   0%|          | 0/20 [00:00<?, ? examples/s]

Dataset prepared: 17 train, 3 test samples


In [7]:
training_args = TrainingArguments(
    output_dir='./gpt2-reviews',
    num_train_epochs=15,
    per_device_train_batch_size=4,
    learning_rate=5e-5,
    weight_decay=0.01,
    warmup_steps=50,
    eval_strategy='epoch',
    logging_steps=10,
    save_strategy='no',
    fp16=torch.cuda.is_available(),
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=split['train'],
    eval_dataset=split['test'],
    data_collator=data_collator,
)

print(">>> Fine-tuning on product review data (15 epochs)...\n")
trainer.train()
print("\nTraining complete!")

>>> Fine-tuning on product review data (15 epochs)...



`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Epoch,Training Loss,Validation Loss
1,No log,3.348063
2,4.057265,3.247350
3,4.057265,3.106139
4,3.319935,2.981021
5,3.319935,2.881039
6,2.706133,2.803390
7,2.706133,2.756230
8,1.918991,2.740822
9,1.918991,2.765902
10,1.234805,2.867215



Training complete!


### Step 4 – Generate Reviews & Compare Before vs After

In [8]:
eval_res = trainer.evaluate()
perplexity = math.exp(eval_res['eval_loss'])
print(f"Perplexity after fine-tuning: {perplexity:.2f}")

print("\n" + "=" * 60)
print("  FINE-TUNED REVIEWS (After Fine-Tuning)")
print("=" * 60)

for p in review_prompts:
    ft_out = generate_text(model, tokenizer, p)
    print(f"\nPrompt    : {p}")
    print(f"  BEFORE  : {baseline[p][:160]}")
    print(f"  AFTER   : {ft_out[:160]}")

print("\n" + "=" * 60)
print("  Component I Complete!")
print("=" * 60)

Perplexity after fine-tuning: 24.74

  FINE-TUNED REVIEWS (After Fine-Tuning)

Prompt    : This product is
  BEFORE  : This product is made from high quality, lightweight stainless steel. If you are looking for something a little more durable, it's a good choice.

Laser Pouch

N
  AFTER   : This product is packed with features that make this purchase even more worthwhile. The quality of this product exceeds expectations and the price points justify

Prompt    : I bought this phone and
  BEFORE  : I bought this phone and I have not used it on a lot of people. I have also not used it on any other people.

The screen was amazing and the sound was amazing. I
  AFTER   : I bought this phone and it handles all my daily chores perfectly. I would definitely buy from this brand again.

Verified purchase: this deal i made this year

Prompt    : The quality of this item
  BEFORE  : The quality of this item in the item description (and if the item is already in stock) will determine how many times

---
# 🍳 COMPONENT II – Recipe Instruction Generator (Food-Tech)

**Scenario:** You are an AI developer at a food-tech startup. Fine-tune GPT-2 to generate step-by-step cooking instructions.

### Step 1 – Reload a Fresh GPT-2 Model

In [9]:
print("Loading fresh GPT-2 for Component II...")

tokenizer2 = GPT2Tokenizer.from_pretrained('gpt2')
model2 = GPT2LMHeadModel.from_pretrained('gpt2')

tokenizer2.pad_token = tokenizer2.eos_token
model2.config.pad_token_id = tokenizer2.eos_token_id

print("Fresh GPT-2 loaded successfully!")

Loading fresh GPT-2 for Component II...


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Fresh GPT-2 loaded successfully!


### Step 2 – Generate Baseline Recipes (Before Fine-Tuning)

In [10]:
recipe_prompts = [
    'To make butter chicken',
    'For pasta carbonara',
    'To prepare a chocolate cake',
]

print("=" * 60)
print("  BASELINE RECIPES (Before Fine-Tuning)")
print("=" * 60)

baseline2 = {}
for p in recipe_prompts:
    baseline2[p] = generate_text(model2, tokenizer2, p)
    print(f"\nPrompt : {p}")
    print(f"Output : {baseline2[p]}")

  BASELINE RECIPES (Before Fine-Tuning)

Prompt : To make butter chicken
Output : To make butter chicken with eggs. I was going to do this with a different recipe, but I was going into a different area of town and was going to have to do something with them. I love a good egg.

Here's how to make your own:

1 pound unsalted butter

1 tsp vanilla extract

8-10 eggs

1 cup all-purpose flour

4 tablespoons unsalted butter

1/2 cup all-purpose flour


Prompt : For pasta carbonara
Output : For pasta carbonara pasta

2 large olives, cut into thin strips

1 large onion, finely chopped

4 small garlic cloves, finely chopped

3 large red bell peppers, finely chopped

1 large green chile, finely chopped

4 cups water

1 tsp dried oregano or ground oregano

salt to taste

4 cups white wine vinegar

1 tbsp olive oil

1 tsp salt

1 cup water

Prompt : To prepare a chocolate cake
Output : To prepare a chocolate cake, you're going to want a cake that is pretty light. You can also use a light brown ca

### Step 3 – Prepare Recipe Dataset & Fine-Tune

In [11]:
recipes = [
    'to make butter chicken start by marinating chicken pieces in yogurt with turmeric chili powder and garam masala for one hour.',
    'heat butter in a pan and fry onions until golden brown then add ginger garlic paste and cook for two minutes.',
    'add tomato puree and cook on low heat for ten minutes until the oil separates from the masala.',
    'add the marinated chicken and cook on medium heat for fifteen minutes until fully cooked.',
    'finish with fresh cream and kasuri methi and serve hot with naan or steamed rice.',
    'for pasta carbonara boil spaghetti in salted water until al dente and reserve half cup of pasta water.',
    'fry diced pancetta in olive oil until crispy and set aside.',
    'whisk together egg yolks parmesan cheese and black pepper in a bowl.',
    'toss the hot pasta with pancetta and remove from heat then quickly stir in the egg mixture.',
    'the residual heat will cook the eggs into a creamy sauce and serve immediately with extra parmesan.',
    'to prepare vegetable stir fry heat sesame oil in a wok on high heat.',
    'add sliced bell peppers broccoli florets and snap peas and toss for three minutes.',
    'pour in soy sauce oyster sauce and a pinch of sugar and stir well.',
    'add minced garlic and ginger and cook for one more minute until fragrant.',
    'serve the stir fry over steamed jasmine rice and garnish with sesame seeds.',
    'for chocolate chip cookies cream together butter and sugar until light and fluffy.',
    'beat in eggs one at a time then add vanilla extract and mix well.',
    'fold in flour baking soda and salt then gently stir in chocolate chips.',
    'scoop dough onto a baking sheet and bake at 180 degrees for twelve minutes until golden.',
    'let cookies cool on the tray for five minutes before transferring to a wire rack.',
]

dataset2 = Dataset.from_dict({'text': recipes})
tok2 = dataset2.map(
    lambda x: tokenizer2(x['text'], truncation=True, max_length=128, padding='max_length'),
    batched=True,
    remove_columns=['text']
)
split2 = tok2.train_test_split(test_size=0.15, seed=42)
collator2 = DataCollatorForLanguageModeling(tokenizer=tokenizer2, mlm=False)

print(f"Recipe dataset prepared: {len(split2['train'])} train, {len(split2['test'])} test samples")

Map:   0%|          | 0/20 [00:00<?, ? examples/s]

Recipe dataset prepared: 17 train, 3 test samples


In [12]:
args2 = TrainingArguments(
    output_dir='./gpt2-recipes',
    num_train_epochs=15,
    per_device_train_batch_size=4,
    learning_rate=5e-5,
    weight_decay=0.01,
    warmup_steps=50,
    eval_strategy='epoch',
    logging_steps=10,
    save_strategy='no',
    fp16=torch.cuda.is_available(),
    report_to="none",
)

trainer2 = Trainer(
    model=model2,
    args=args2,
    train_dataset=split2['train'],
    eval_dataset=split2['test'],
    data_collator=collator2,
)

print(">>> Fine-tuning on recipe data (15 epochs)...\n")
trainer2.train()
print("\nTraining complete!")

>>> Fine-tuning on recipe data (15 epochs)...



Epoch,Training Loss,Validation Loss
1,No log,3.403592
2,3.926337,3.264174
3,3.926337,3.105294
4,3.443312,3.021800
5,3.443312,2.981154
6,2.728940,2.930787
7,2.728940,2.897742
8,2.090287,2.897580
9,2.090287,2.992910
10,1.577087,3.140069



Training complete!


### Step 4 – Generate Recipes & Compare Before vs After

In [13]:
eval2 = trainer2.evaluate()
perplexity2 = math.exp(eval2['eval_loss'])
print(f"Perplexity after fine-tuning: {perplexity2:.2f}")

print("\n" + "=" * 60)
print("  FINE-TUNED RECIPES (After Fine-Tuning)")
print("=" * 60)

for p in recipe_prompts:
    ft = generate_text(model2, tokenizer2, p)
    print(f"\nPrompt    : {p}")
    print(f"  BEFORE  : {baseline2[p][:160]}")
    print(f"  AFTER   : {ft[:160]}")

print("\n" + "=" * 60)
print("  Component II Complete!")
print("=" * 60)
print("\n✅ Lab 11 Complete! Both components finished successfully.")

Perplexity after fine-tuning: 36.75

  FINE-TUNED RECIPES (After Fine-Tuning)

Prompt    : To make butter chicken
  BEFORE  : To make butter chicken with eggs. I was going to do this with a different recipe, but I was going into a different area of town and was going to have to do some
  AFTER   : To make butter chicken add turmeric chili powder and garam masala and cook for one more minute until the chicken is soft. Add chicken pieces and cook for two mi

Prompt    : For pasta carbonara
  BEFORE  : For pasta carbonara pasta

2 large olives, cut into thin strips

1 large onion, finely chopped

4 small garlic cloves, finely chopped

3 large red bell peppers,
  AFTER   : For pasta carbonara boil spaghetti in salted water until al dente and reserve half cup of pasta water. Drain pasta and remove from heat. Add reserved pasta wate

Prompt    : To prepare a chocolate cake
  BEFORE  : To prepare a chocolate cake, you're going to want a cake that is pretty light. You can also use a light brow